# WeaviateVectorStore Test Notebook

This notebook demonstrates how to use the WeaviateVectorStore class to:
1. Create a tenant
2. Index a document with chunks
3. Query for similar chunks

## Prerequisites

**IMPORTANT**: Before running this notebook:

1. **Weaviate must be running** with the OpenAI API key configured:
   ```bash
   # From project root
   docker-compose down
   docker-compose up -d
   ```

In [1]:
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

# Add the src directory to Python path
src_path = Path.cwd().parent / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from doc_chat.rag.weaviate_vector_store import (
    WeaviateVectorStore,
    Document,
    DocumentChunk
)

## Clean Up Existing Collections

First, let's delete any existing collections to ensure they're created with the correct multi-tenancy configuration

In [2]:
import weaviate

# Connect to Weaviate and delete existing collections
client = weaviate.connect_to_local(host="localhost", port=8080)

# Delete collections if they exist
for collection_name in ["Document", "DocumentChunk"]:
    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)
        print(f"✓ Deleted existing '{collection_name}' collection")
    else:
        print(f"  '{collection_name}' collection doesn't exist (skip)")

client.close()
print("\n✓ Cleanup complete")

✓ Deleted existing 'Document' collection
✓ Deleted existing 'DocumentChunk' collection

✓ Cleanup complete


## Initialize WeaviateVectorStore

Make sure Weaviate is running locally on port 8080 (e.g., via `docker-compose up`)

In [3]:
# Initialize the vector store
vector_store = WeaviateVectorStore(embedding_model='text-embedding-3-small')
print("✓ WeaviateVectorStore initialized")
print(f"✓ Client ready: {vector_store.client.is_ready()}")

/Users/patrick/projects/doc-chat/api/venv/lib/python3.13/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


✓ WeaviateVectorStore initialized
✓ Client ready: True


## Create a Tenant

Multi-tenancy allows us to isolate data per user

In [4]:
# Create a test user tenant
test_user_id = f"test_user_{uuid4().hex[:8]}"
print(f"Creating tenant for user: {test_user_id}")

vector_store.create_tenant(test_user_id)
print(f"✓ Tenant created for {test_user_id}")

Creating tenant for user: test_user_84ba33eb
✓ Tenant created for test_user_84ba33eb


## Create Sample Document and Chunks

We'll create a sample document about Python programming with several chunks

In [5]:
# Create a sample document
doc_id = f"doc_{uuid4().hex[:8]}"
document = Document(
    doc_id=doc_id,
    file_name="python_guide.pdf",
    created_at=datetime.now(timezone.utc),
    num_pages=3
)

# Create sample chunks with different content
chunks = [
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=1,
        page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms including procedural, object-oriented, and functional programming.",
        created_at=datetime.now(timezone.utc)
    ),
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=1,
        page_content="Python's syntax emphasizes code readability with significant whitespace. The language provides constructs intended to enable writing clear programs on both small and large scales.",
        created_at=datetime.now(timezone.utc)
    ),
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=2,
        page_content="Python has a comprehensive standard library that supports many common programming tasks such as connecting to web servers, reading and writing files, and working with data.",
        created_at=datetime.now(timezone.utc)
    ),
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=2,
        page_content="Popular Python frameworks include Django and Flask for web development, NumPy and Pandas for data analysis, and TensorFlow and PyTorch for machine learning applications.",
        created_at=datetime.now(timezone.utc)
    ),
    DocumentChunk(
        chunk_id=f"chunk_{uuid4().hex[:8]}",
        doc_id=doc_id,
        page_number=3,
        page_content="Python's dynamic typing and automatic memory management make it easy to learn for beginners while remaining powerful enough for complex applications. It's widely used in web development, data science, automation, and artificial intelligence.",
        created_at=datetime.now(timezone.utc)
    )
]

print(f"✓ Created document '{document.file_name}' with {len(chunks)} chunks")
for i, chunk in enumerate(chunks, 1):
    print(f"  Chunk {i} (page {chunk.page_number}): {chunk.page_content[:60]}...")

✓ Created document 'python_guide.pdf' with 5 chunks
  Chunk 1 (page 1): Python is a high-level, interpreted programming language kno...
  Chunk 2 (page 1): Python's syntax emphasizes code readability with significant...
  Chunk 3 (page 2): Python has a comprehensive standard library that supports ma...
  Chunk 4 (page 2): Popular Python frameworks include Django and Flask for web d...
  Chunk 5 (page 3): Python's dynamic typing and automatic memory management make...


## Index the Document

Now we'll index the document and its chunks in Weaviate

In [6]:
# Index the document and chunks
print(f"Indexing document {doc_id}...")
document_uuid = vector_store.index_document(
    user_id=test_user_id,
    document=document,
    chunks=chunks
)

print(f"✓ Document indexed successfully!")
print(f"  Document UUID: {document_uuid}")
print(f"  Indexed {len(chunks)} chunks with embeddings")

Indexing document doc_364b5028...
✓ Document indexed successfully!
  Document UUID: 4dbecff9-b0db-4259-ab68-1b3534271afd
  Indexed 5 chunks with embeddings


## Query for Similar Chunks

Let's run some queries to retrieve relevant chunks based on semantic similarity

In [7]:
# Query 1: Search for information about machine learning
query1 = "machine learning frameworks"
print(f"Query: '{query1}'\n")

results1 = vector_store.retrieve_documents(
    user_id=test_user_id,
    doc_id=doc_id,
    query=query1,
    k=2
)

print(f"Found {len(results1)} relevant chunks:\n")
for i, chunk in enumerate(results1, 1):
    print(f"Result {i}:")
    print(f"  Page: {chunk.page_number}")
    print(f"  Content: {chunk.page_content}")
    print()

Query: 'machine learning frameworks'

Found 2 relevant chunks:

Result 1:
  Page: 2
  Content: Popular Python frameworks include Django and Flask for web development, NumPy and Pandas for data analysis, and TensorFlow and PyTorch for machine learning applications.

Result 2:
  Page: 3
  Content: Python's dynamic typing and automatic memory management make it easy to learn for beginners while remaining powerful enough for complex applications. It's widely used in web development, data science, automation, and artificial intelligence.



In [8]:
# Query 2: Search for information about Python syntax
query2 = "code readability and syntax"
print(f"Query: '{query2}'\n")

results2 = vector_store.retrieve_documents(
    user_id=test_user_id,
    doc_id=doc_id,
    query=query2,
    k=2
)

print(f"Found {len(results2)} relevant chunks:\n")
for i, chunk in enumerate(results2, 1):
    print(f"Result {i}:")
    print(f"  Page: {chunk.page_number}")
    print(f"  Content: {chunk.page_content}")
    print()

Query: 'code readability and syntax'

Found 2 relevant chunks:

Result 1:
  Page: 1
  Content: Python's syntax emphasizes code readability with significant whitespace. The language provides constructs intended to enable writing clear programs on both small and large scales.

Result 2:
  Page: 1
  Content: Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms including procedural, object-oriented, and functional programming.



In [9]:
# Query 3: Search for information about web development
query3 = "web servers and files"
print(f"Query: '{query3}'\n")

results3 = vector_store.retrieve_documents(
    user_id=test_user_id,
    doc_id=doc_id,
    query=query3,
    k=2
)

print(f"Found {len(results3)} relevant chunks:\n")
for i, chunk in enumerate(results3, 1):
    print(f"Result {i}:")
    print(f"  Page: {chunk.page_number}")
    print(f"  Content: {chunk.page_content}")
    print()

Query: 'web servers and files'

Found 2 relevant chunks:

Result 1:
  Page: 2
  Content: Python has a comprehensive standard library that supports many common programming tasks such as connecting to web servers, reading and writing files, and working with data.

Result 2:
  Page: 2
  Content: Popular Python frameworks include Django and Flask for web development, NumPy and Pandas for data analysis, and TensorFlow and PyTorch for machine learning applications.



## Cleanup (Optional)

Close the Weaviate client connection when done

In [10]:
# Close the client connection
vector_store.client.close()
print("✓ Weaviate client connection closed")

✓ Weaviate client connection closed
